# DRL v5.1 — PQN experiments on Colab Pro

Run cells top to bottom. Every sweep is **resumable**: results are written to Google Drive one config at a time; if the session disconnects, re-run the same cell and finished configs are skipped. Keep this tab open (Colab Pro has no background execution).

Upload `drl_v5_code.zip` to `MyDrive/drl_v5/` first.

**Protocol (v5.1):** every run except `public_gymnax` trains and evaluates with sticky actions p=0.1 and a 10000-step episode cap (`sweep.py --protocol`). Each eval point logs both the public protocol (deterministic, cap 1000) and the robust protocol (primary metric). Reason: the CPU pilot showed the public evaluation saturates — every greedy episode hit the 1000-step cap and all eval episodes were identical.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/drl_v5'
!mkdir -p $OUT && unzip -o -q $OUT/drl_v5_code.zip -d /content/code && ls /content/code
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Install
Option A pins the stack the code was tested with. If it fails to install or `jax.devices()` shows no GPU, restart the runtime and use option B (Colab's own JAX + gymnax 1.0).

**Pick one option before P0 and never switch within a stage**: random streams differ between JAX versions (each output JSON records `versions`), so seeds of a stage resumed under another runtime would not be comparable.

In [ ]:
# Option A (tested stack)
!pip -q install -U "jax[cuda12]==0.4.38" flax==0.10.2 optax==0.2.4 chex==0.1.88 gymnax==0.0.6 six imageio imageio-ffmpeg pillow
# Option B (fallback): !pip -q install gymnax==1.0.0 imageio imageio-ffmpeg
import jax; print(jax.__version__, jax.devices())

## 2. Smoke test and timing
The timing commands measure seconds per 0.5M env steps for E=2 and E=8 **including the robust evaluation** (cap 10000); use them to re-plan the budget.

In [ ]:
%cd /content/code
!python test_v5.py | tail -8
PROTO = '--sticky 0.1 --max_steps_in_episode 10000 --eval_cap 10000 --eval_sticky 0.1 --bootstrap_timeout'
!python -W ignore pqn_v5.py --steps 8192 --seeds 1 --eval_every 8192 --eval_episodes 8 {PROTO} --out /content/smoke.json | tail -2
!python -W ignore pqn_v5.py --steps 1e6 --seeds 10 --epochs 2 --eval_every 5e5 {PROTO} --out /content/t2.json | tail -3
!python -W ignore pqn_v5.py --steps 1e6 --seeds 10 --epochs 8 --eval_every 5e5 {PROTO} --out /content/t8.json | tail -3

## 3. P0 — protocol + correction ladder (5 configs × 10 seeds × 10M steps)

In [ ]:
!python sweep.py P0 --outdir $OUT
for m in ['robust_return', 'greedy_return', 'train_return']:
    !python analyze.py $OUT/P0 --metric {m} --ref public --plot $OUT/P0_{m}.png

## 4. P1 — screening on development seeds (19 configs × 3 seeds)
Both code fixes + timeout bootstrapping are the base of every arm. **Write down the selection rule before looking at P1** (default: highest mean robust AUC; if its margin over `E2_none` is below the between-seed SD, P2 confirms only the fixed PQN).

In [ ]:
!python sweep.py P1 --outdir $OUT
!python analyze.py $OUT/P1 --metric robust_return --ref E2_none --plot $OUT/P1.png
!python analyze.py $OUT/P1 --metric train_return --ref E2_none

## 5. P2 — confirmation on fresh seeds (freeze E* and control first)

In [ ]:
BEST_E = 8                                   # frozen from P1 (frozen_P2.json)
BEST_CTRL = '--norm_ctrl np --lr 2.5e-4'     # frozen from P1 (control + its learning rate)
!python sweep.py P2 --outdir $OUT --best_E $BEST_E --best_ctrl "$BEST_CTRL"
!python analyze.py $OUT/P2 --metric robust_return --ref fixed_E2 --json_out $OUT/P2_vs_fixed.json --plot $OUT/P2.png
!python analyze.py $OUT/P2 --metric robust_return --ref public --json_out $OUT/P2_vs_public.json

## 6. (Optional) P3 — robustness on Breakout / Asterix / Freeway

In [ ]:
!python sweep.py P3 --outdir $OUT --best_E $BEST_E --best_ctrl "$BEST_CTRL"
# 5 seeds: descriptive only (the exact sign-flip test cannot reach p < 0.05 with n = 5)
for game in ['Breakout', 'Asterix', 'Freeway']:
    !python analyze.py $OUT/P3 --filter {game}_ --metric robust_return --ref {game}_fixed_E2

## 7. Demo videos (SpaceInvaders, sticky 0.1, same env seeds, median of 9 full episodes)
1. Learning progression of the final method (1M / 3M / 10M) — valid whatever the comparison shows.
2. Public PQN vs ours, and fixed PQN (E=2) vs ours, at 10M steps.

In [ ]:
import glob, json, numpy as np
def g1(pat):
    hits = sorted(glob.glob(f'{OUT}/P2/{pat}'))
    assert len(hits) == 1, (pat, hits)          # exactly one file, never a snapshot by accident
    return hits[0]
FINAL = '__*[0-9a-f]_params.pkl'                # final params end in '<hash>_params.pkl'; snapshots in '_ckptXM_params.pkl'
pub, fix, ours = g1('public' + FINAL), g1('fixed_E2' + FINAL), g1('ours_E*' + FINAL)
o1, o3 = g1('ours_E*_ckpt1M_params.pkl'), g1('ours_E*_ckpt3M_params.pkl')
def median_seed(params_path):                   # pre-registered: the seed whose robust AUC is the median of the config
    d = json.load(open(params_path.replace('_params.pkl', '.json')))
    auc = np.array([e['robust_return'] for e in d['evals']]).mean(0)
    order = np.argsort(auc); lo, hi = order[len(auc) // 2 - 1], order[len(auc) // 2]
    return int(lo if abs(auc[lo] - auc.mean()) <= abs(auc[hi] - auc.mean()) else hi) if len(auc) % 2 == 0 else int(order[len(auc) // 2])
SP, SF, SO = median_seed(pub), median_seed(fix), median_seed(ours)
print('median-AUC seed index  public:', SP, ' fixed_E2:', SF, ' ours:', SO)
!python -W ignore render_demo.py $OUT/demo_progression.mp4 --seed_indices $SO,$SO,$SO --panel "$o1" "1M steps" --panel "$o3" "3M steps" --panel "$ours" "10M steps" --frame_png $OUT/demo_progression.png

In [ ]:
!python -W ignore render_demo.py $OUT/demo_public_vs_ours.mp4 --seed_indices $SP,$SO --panel "$pub" "Public PQN" --panel "$ours" "Ours" --scale255 0
!python -W ignore render_demo.py $OUT/demo_fixed_vs_ours.mp4 --seed_indices $SF,$SO --panel "$fix" "Fixed PQN (E=2)" --panel "$ours" "Ours"